In [1]:
import pandas as pd
import numpy as np
from pathlib import Path



df = pd.read_csv("data/processed/videos_clean.csv")


print("Shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

Shape: (500, 23)

Columns:
['ID', 'Category', 'Channel', 'Title', 'URL', 'Duration', 'Views', 'Explanation_Type', 'Visuals', 'Examples', 'Analogies', 'Real_Life', 'Quality', 'Creator_Takeaways', 'duration_seconds', 'views_numeric', 'combined_text', 'has_analogy', 'has_example', 'has_visual', 'has_math', 'has_code', 'has_application']


In [2]:
text_columns = [
    "Category",
    "Channel",
    "Title",
    "Explanation_Type",
    "Visuals",
    "Examples",
    "Analogies",
    "Real_Life",
    "Quality",
    "Creator_Takeaways"
]

for col in text_columns:
    if col in df.columns:
        df[col] = df[col].fillna("").astype(str).str.strip()

print("Text columns cleaned.")

Text columns cleaned.


In [3]:
def duration_to_seconds(duration):
    if pd.isna(duration):
        return np.nan

    parts = str(duration).strip().split(":")

    try:
        if len(parts) == 2:
            minutes, seconds = map(int, parts)
            return minutes * 60 + seconds

        if len(parts) == 3:
            hours, minutes, seconds = map(int, parts)
            return hours * 3600 + minutes * 60 + seconds

    except ValueError:
        return np.nan

    return np.nan

df["duration_seconds"] = df["Duration"].apply(duration_to_seconds)

print(df[["Duration", "duration_seconds"]].head())

  Duration  duration_seconds
0    18:40              1120
1    20:33              1233
2    12:45               765
3     6:36               396
4     6:05               365


In [4]:
def convert_views(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().replace("+", "").replace(",", "")

    try:
        if value.endswith("B"):
            return float(value[:-1]) * 1_000_000_000
        if value.endswith("M"):
            return float(value[:-1]) * 1_000_000
        if value.endswith("K"):
            return float(value[:-1]) * 1_000

        return float(value)

    except ValueError:
        return np.nan

df["views_numeric"] = df["Views"].apply(convert_views)

print(df[["Views", "views_numeric"]].head())

    Views  views_numeric
0  24.2M+     24200000.0
1   9.5M+      9500000.0
2   1.6M+      1600000.0
3   1.7M+      1700000.0
4   1.5M+      1500000.0


In [5]:
df["log_views"] = np.log1p(df["views_numeric"])

print(df[["views_numeric", "log_views"]].head())

   views_numeric  log_views
0     24200000.0  17.001863
1      9500000.0  16.066802
2      1600000.0  14.285515
3      1700000.0  14.346139
4      1500000.0  14.220976


In [6]:
df["has_analogy"] = (df["Analogies"].str.len() > 10).astype(int)
df["has_example"] = (df["Examples"].str.len() > 10).astype(int)
df["has_visual"] = (df["Visuals"].str.len() > 10).astype(int)
df["has_application"] = (df["Real_Life"].str.len() > 10).astype(int)

df["has_math"] = df["combined_text"].str.contains(
    r"formula|equation|mathematical|math|calculus|derivation|geometry|gradient",
    case=False,
    regex=True,
    na=False
).astype(int)

df["has_code"] = df["combined_text"].str.contains(
    r"code|python|pytorch|tensorflow|programming|implementation|terminal|IDE",
    case=False,
    regex=True,
    na=False
).astype(int)

feature_columns = [
    "has_analogy",
    "has_example",
    "has_visual",
    "has_application",
    "has_math",
    "has_code"
]

print(df[feature_columns].sum())

has_analogy        500
has_example        500
has_visual         500
has_application    500
has_math           197
has_code           316
dtype: int64


In [7]:
df["combined_text"] = (
    "Title: " + df["Title"] +
    "\nCategory: " + df["Category"] +
    "\nExplanation type: " + df["Explanation_Type"] +
    "\nVisuals: " + df["Visuals"] +
    "\nExamples: " + df["Examples"] +
    "\nAnalogies: " + df["Analogies"] +
    "\nReal-life applications: " + df["Real_Life"] +
    "\nQuality description: " + df["Quality"] +
    "\nCreator takeaways: " + df["Creator_Takeaways"]
)

print(df["combined_text"].iloc[0])

Title: But what is a neural network? | Deep learning chapter 1
Category: Machine Learning (ML)
Explanation type: Mathematical & Geometric Intuition
Visuals: Custom Manim animations; dynamic 28x28 pixel grid activations; colored weight synapses (green/red); layered node transformations.
Examples: MNIST handwritten digit recognition focusing on handwritten '3's vs other digits.
Analogies: Biological neurons firing vs mathematical dials/weights (0 to 1); layered abstraction hierarchy (pixels -> edges -> loops -> digits).
Real-life applications: Human visual cortex processing; speech recognition acoustic decomposition; YouTube recommendation algorithms.
Quality description: Pristine studio acoustics; calm, deliberate cadence (~130 wpm); subtle musical scoring; 1080p razor-sharp vector graphics.
Creator takeaways: Hook with an intuitive mystery; never introduce formulas before visual intuition; break black boxes into modular components.


In [8]:
df["text_length"] = df["combined_text"].str.len()
df["word_count"] = df["combined_text"].str.split().str.len()

print(df[["ID", "text_length", "word_count"]].head())

        ID  text_length  word_count
0  VID-001          945         119
1  VID-002          795          99
2  VID-003          773          93
3  VID-004          760          90
4  VID-005          715          87


In [9]:
engineered_columns = [
    "ID",
    "duration_seconds",
    "views_numeric",
    "log_views",
    "has_analogy",
    "has_example",
    "has_visual",
    "has_application",
    "has_math",
    "has_code",
    "text_length",
    "word_count"
]

display(df[engineered_columns].head(10))

print("\nMissing values in engineered features:")
print(df[engineered_columns].isna().sum())

,ID,duration_seconds,views_numeric,log_views,has_analogy,has_example,has_visual,has_application,has_math,has_code,text_length,word_count
0,VID-001,1120,24200000.0,17.001863,1,1,1,1,1,0,945,119
1,VID-002,1233,9500000.0,16.066802,1,1,1,1,1,0,795,99
2,VID-003,765,1600000.0,14.285515,1,1,1,1,0,1,773,93
3,VID-004,396,1700000.0,14.346139,1,1,1,1,0,1,760,90
4,VID-005,365,1500000.0,14.220976,1,1,1,1,0,1,715,87
5,VID-006,433,979000.0,13.794288,1,1,1,1,1,1,738,94
6,VID-007,1088,1300000.0,14.077876,1,1,1,1,1,0,783,94
7,VID-008,528,2800000.0,14.845130,1,1,1,1,0,1,667,78
8,VID-009,594,1500000.0,14.220976,1,1,1,1,0,0,767,99
9,VID-010,510,2200000.0,14.603968,1,1,1,1,0,1,731,87



Missing values in engineered features:
ID                  0
duration_seconds    0
views_numeric       0
log_views           0
has_analogy         0
has_example         0
has_visual          0
has_application     0
has_math            0
has_code            0
text_length         0
word_count          0
dtype: int64


In [10]:
numeric_columns = [
    "duration_seconds",
    "views_numeric",
    "log_views",
    "text_length",
    "word_count"
]

display(df[numeric_columns].describe())

,duration_seconds,views_numeric,log_views,text_length,word_count
count,500.000000,5.000000e+02,500.000000,500.000000,500.000000
mean,20514.748000,8.902831e+05,11.899010,838.902000,107.564000
std,47978.474076,2.745575e+06,2.037287,79.554524,12.486061
min,60.000000,7.300000e+01,4.304065,661.000000,78.000000
25%,482.750000,4.000000e+04,10.596660,780.000000,98.000000
50%,768.000000,1.510000e+05,11.925042,833.000000,107.000000
75%,1351.500000,5.907500e+05,13.289147,890.250000,116.000000
max,215280.000000,3.830000e+07,17.460960,1098.000000,146.000000
